# Fraud Risk Summary — SQL

SQL-cell equivalent of `fraud_risk_summary.ipynb`, using `%%sql` cells
instead of the DataFrame API. Same result: reads `payments.card_transactions`
and persists `payments.fraud_risk_summary_v1`.

Requires the bronze table to already exist — run
`data-generators/card_transactions_ingest.ipynb` first.

`%%sql` here follows Databricks-style notebook cell magic. If running
outside a platform that provides it natively, install and load
`sparksql-magic` first: `pip install sparksql-magic` then
`%load_ext sparksql_magic` (bound to the `spark` session created below).

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Transform + persist: fraud risk by amount bucket and hour

In [ ]:
%%sql
DROP TABLE IF EXISTS payments.fraud_risk_summary_v1;

CREATE TABLE payments.fraud_risk_summary_v1 AS
WITH bucketed AS (
    SELECT
        CAST(CAST(Time AS LONG) % 86400 / 3600 AS INT) AS hour_of_day,
        CASE
            WHEN Amount < 10 THEN '0-10'
            WHEN Amount < 50 THEN '10-50'
            WHEN Amount < 100 THEN '50-100'
            WHEN Amount < 500 THEN '100-500'
            ELSE '500+'
        END AS amount_bucket,
        CASE WHEN Class = '1' THEN 1 ELSE 0 END AS is_fraud,
        Amount
    FROM payments.card_transactions
)
SELECT
    amount_bucket,
    hour_of_day,
    COUNT(*) AS transaction_count,
    SUM(is_fraud) AS fraud_count,
    ROUND(SUM(CASE WHEN is_fraud = 1 THEN Amount ELSE 0.0 END), 2) AS total_flagged_amount,
    ROUND(SUM(is_fraud) * 100.0 / COUNT(*), 3) AS fraud_rate_pct
FROM bucketed
GROUP BY amount_bucket, hour_of_day
ORDER BY fraud_count DESC

## Verify

In [ ]:
%%sql
SELECT COUNT(*) AS total_buckets FROM payments.fraud_risk_summary_v1

In [ ]:
%%sql
SELECT * FROM payments.fraud_risk_summary_v1
ORDER BY fraud_count DESC
LIMIT 10